# 06 · Monthly Field & Product Update
Monthly digest of customer conversation themes for the field and product teams.
Aggregates Gong insights, use case stage movement, and new account engagement.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
report_month = datetime.date(2026, 6, 1)   # first day of the month
# ─────────────────────────────────────────────────────────
month_start = report_month.replace(day=1)
if month_start.month == 12:
    month_end = month_start.replace(year=month_start.year+1, month=1, day=1) - datetime.timedelta(days=1)
else:
    month_end = month_start.replace(month=month_start.month+1, day=1) - datetime.timedelta(days=1)
ms, me = str(month_start), str(min(month_end, today))
print(f"Monthly report: {ms} → {me}  ({month_start.strftime('%B %Y')})")


## Monthly Engagement Summary

In [ ]:
%%sql -r monthly_kpi
SELECT
    COUNT(*)                                                    AS total_meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)                               AS unique_accounts,
    COUNT(DISTINCT CUSTOMER)                                    AS unique_companies,
    COUNT(DISTINCT MEETING_SE_NAME)                             AS specialists,
    SUM(IFF(SUMMARY IS NOT NULL,1,0))                           AS gong_summaries,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))                   AS with_use_case
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ms}}' AND '{{me}}'

In [ ]:
monthly_kpi = monthly_kpi.to_pandas() if hasattr(monthly_kpi, "to_pandas") else monthly_kpi
r = monthly_kpi.iloc[0]
items = [
    (int(r['TOTAL_MEETINGS'] or 0),   'Total Meetings',         '#29B5E8'),
    (int(r['UNIQUE_ACCOUNTS'] or 0),  'Unique Accounts',        '#00A9CE'),
    (int(r['UNIQUE_COMPANIES'] or 0), 'Companies',              '#36B37E'),
    (int(r['SPECIALISTS'] or 0),      'Active Specialists',     '#FF8B00'),
    (int(r['GONG_SUMMARIES'] or 0),   'Gong Summaries',         '#9B59B6'),
]
html = '<div style="display:flex;gap:16px;flex-wrap:wrap;margin:12px 0">'
for v,t,c in items:
    html += (f'<div style="background:#f0f8ff;border-left:4px solid {c};padding:16px 24px;border-radius:4px;min-width:140px">'
             f'<div style="font-size:32px;font-weight:bold;color:{c}">{v}</div>'
             f'<div style="font-size:11px;color:#555;margin-top:4px">{t}</div></div>')
html += '</div>'
display(HTML(html))


## AI — Top Customer Themes This Month

In [ ]:
%%sql -r month_summaries
SELECT MEETING_DATE, CUSTOMER, MEETING_SE_NAME, SUMMARY, NEXT_STEPS
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_DATE BETWEEN '{{ms}}' AND '{{me}}'
  AND SUMMARY IS NOT NULL AND TRIM(SUMMARY) != ''
ORDER BY MEETING_DATE

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
month_summaries = month_summaries.to_pandas() if hasattr(month_summaries, "to_pandas") else month_summaries
texts = "\n---\n".join(
    f"[{r['MEETING_DATE']} - {r['MEETING_SE_NAME']} @ {r['CUSTOMER']}]\n{r['SUMMARY']}"
    for _, r in month_summaries.iterrows() if r['SUMMARY']
)
if texts.strip():
    prompt = (
        f"You are preparing a monthly customer insights report for {month_start.strftime('%B %Y')} "
        "for a Snowflake specialist team's field and product partners.\n\n"
        "Based on these customer meeting summaries, produce a structured report covering:\n\n"
        "**1. TOP CUSTOMER THEMES** (5-7 themes, with supporting evidence from the summaries)\n"
        "**2. PRODUCT/FEATURE REQUESTS** (what are customers asking for that we don't have)\n"
        "**3. COMPETITIVE SIGNALS** (any Databricks, dbt, other competitor mentions)\n"
        "**4. SUCCESS STORIES** (positive outcomes, use cases going well)\n"
        "**5. RISK SIGNALS** (customers expressing concerns, delays, or blockers)\n"
        "**6. RECOMMENDATIONS FOR FIELD/PRODUCT** (3-5 actionable recommendations)\n\n"
        "Use specific customer names and quotes where available. Keep each section concise.\n\n"
        "Meeting summaries:\n" + texts[:7000]
    )
    result = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    month_label = month_start.strftime('%B %Y')
    print(f"=== {month_label} Customer Insights Report ===\n")
    print(result)
else:
    print("No Gong summaries available for this month.")

## Account Engagement by Segment/Industry

In [ ]:
%%sql -r acct_detail
SELECT
    a.CUSTOMER,
    a.SF_ACCOUNT_ID,
    COUNT(*) AS meetings,
    MAX(a.MEETING_DATE) AS last_touch,
    MAX(a.USE_CASE_NAME) AS use_case,
    MAX(a.USE_CASE_TAGGED_IN_SF) AS uc_status,
    d.ACCOUNT_INDUSTRY,
    d.ACCOUNT_SEGMENT
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
LEFT JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.SF_ACCOUNT_ID = d.ACCOUNT_ID
WHERE a.MEETING_DATE BETWEEN '{{ms}}' AND '{{me}}'
  AND a.CUSTOMER IS NOT NULL
GROUP BY 1,2,7,8 ORDER BY 3 DESC LIMIT 40

In [ ]:
acct_detail = acct_detail.to_pandas() if hasattr(acct_detail, "to_pandas") else acct_detail
html_table(acct_detail)

## Use Case Pipeline Movement This Month

In [ ]:
%%sql -r pipeline_movement
SELECT
    d.USE_CASE_STAGE,
    d.STAGE_NUMBER,
    COUNT(DISTINCT d.USE_CASE_ID) AS use_cases,
    ROUND(SUM(d.USE_CASE_EACV)/1e6,2) AS eacv_m,
    SUM(IFF(d.LAST_MODIFIED_DATE BETWEEN '{{ms}}' AND '{{me}}',1,0)) AS updated_this_month
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
WHERE a.USER_ID IN ('{{team_ids_str}}')
  AND d.IS_LOST = FALSE AND d.STAGE_NUMBER BETWEEN 1 AND 6
GROUP BY 1,2 ORDER BY 2

In [ ]:
pipeline_movement = pipeline_movement.to_pandas() if hasattr(pipeline_movement, "to_pandas") else pipeline_movement
html_table(pipeline_movement)

## New Accounts Engaged This Month (First Touch)

In [ ]:
%%sql -r new_accounts
SELECT
    a.CUSTOMER,
    a.SF_ACCOUNT_ID,
    MIN(a.MEETING_DATE) AS first_touch,
    a.MEETING_SE_NAME   AS specialist,
    a.OPPORTUNITY_NAME,
    a.USE_CASE_TAGGED_IN_SF AS uc_status
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES a
WHERE a.MEETING_DATE BETWEEN '{{ms}}' AND '{{me}}'
  AND a.CUSTOMER IS NOT NULL
  AND NOT EXISTS (
      SELECT 1 FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES b
      WHERE b.SF_ACCOUNT_ID = a.SF_ACCOUNT_ID
        AND b.MEETING_DATE < '{{ms}}'
  )
GROUP BY 1,2,4,5,6
ORDER BY first_touch

In [ ]:
new_accounts = new_accounts.to_pandas() if hasattr(new_accounts, "to_pandas") else new_accounts
print(f"New accounts engaged this month: {len(new_accounts)}")
html_table(new_accounts)